### Imports

In [8]:
import pandas as pd
import numpy as np
import pickle

from pathlib import Path

## Project Path

In [9]:
project = Path(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx"
)

print(project.exists())

True


In [10]:
ml_dir = project / "data" / "ml_ready"

ml_dir.mkdir(exist_ok=True)

print(ml_dir)

C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx\data\ml_ready


### Load All Edge Files

In [11]:
cve_cwe = pd.read_csv(
    project / "data" / "processed" / "cve_cwe_edges.csv"
)

cwe_capec = pd.read_csv(
    project / "data" / "processed" / "cwe_capec_edges.csv"
)

capec_attack = pd.read_csv(
    project / "data" / "processed" / "capec_attack_edges.csv"
)

attack_mitigation = pd.read_csv(
    project / "data" / "processed" / "attack_mitigation_edges.csv"
)

software_edges = pd.read_csv(
    project / "data" / "processed" / "software_edges.csv"
)

### Verify Counts

In [12]:
print("CVE → CWE:", len(cve_cwe))

print("CWE → CAPEC:", len(cwe_capec))

print("CAPEC → ATT&CK:", len(capec_attack))

print("ATT&CK → Mitigation:", len(attack_mitigation))

print("CVE → Software:", len(software_edges))

CVE → CWE: 2296
CWE → CAPEC: 1212
CAPEC → ATT&CK: 5845
ATT&CK → Mitigation: 1448
CVE → Software: 4772


### Merge Everything

In [13]:
edges = pd.concat(
    [
        cve_cwe,
        cwe_capec,
        capec_attack,
        attack_mitigation,
        software_edges
    ],
    ignore_index=True
)

In [14]:
edges.head()

,source,relation,target
0,CVE-2026-12212,has_weakness,CWE-266
1,CVE-2026-12212,has_weakness,CWE-284
2,CVE-2026-12213,has_weakness,CWE-266
3,CVE-2026-12213,has_weakness,CWE-285
4,CVE-2026-12214,has_weakness,CWE-693


### Total Edge Count

In [15]:
print("Total edges:", len(edges))

Total edges: 15573


### All Unique Nodes

In [16]:
all_nodes = sorted(
    set(edges["source"])
    .union(
        set(edges["target"])
    )
)

len(all_nodes)

7502

### Inspect Nodes

In [17]:
all_nodes[:20]

['10Web:Form Maker by 10Web',
 '10web:Form Maker by 10Web – Mobile-Friendly Drag & Drop Contact Form Builder',
 '2download:2Download Connector for 2DL Hosted Checkout',
 '404-redirection-manager:404 Redirection Manager',
 '@remix-run:server-runtime',
 'A WP Life:Webenvo',
 'AA-Team:Premium Age Verification / Restriction for WordPress',
 'ACPT:ACPT (Pro) - Custom Post Types Plugin for WordPress',
 'ANSSI:DFIR-ORC',
 'AOMEI:Backupper',
 'AOMEI:Dynamic Disk Manager',
 'AOMEI:Partition Assistant',
 'ASUS:Armoury Crate',
 'AVer:PTC115',
 'AVer:PTC115+',
 'AVer:PTC500+',
 'AVer:PTC500S',
 'AVideo:AVideo',
 'AWS:Kiro IDE',
 'AWS:bedrock-agentcore']

### Create node2id

In [18]:
## as ML models cannot use strings

node2id = {}

for idx, node in enumerate(all_nodes):

    node2id[node] = idx

### Check Mapping

In [19]:
list(node2id.items())[:10]

[('10Web:Form Maker by 10Web', 0),
 ('10web:Form Maker by 10Web – Mobile-Friendly Drag & Drop Contact Form Builder',
  1),
 ('2download:2Download Connector for 2DL Hosted Checkout', 2),
 ('404-redirection-manager:404 Redirection Manager', 3),
 ('@remix-run:server-runtime', 4),
 ('A WP Life:Webenvo', 5),
 ('AA-Team:Premium Age Verification / Restriction for WordPress', 6),
 ('ACPT:ACPT (Pro) - Custom Post Types Plugin for WordPress', 7),
 ('ANSSI:DFIR-ORC', 8),
 ('AOMEI:Backupper', 9)]

### Create Relation IDs

In [20]:
edges["relation"].unique()

array(['has_weakness', 'mapped_to', 'maps_to', 'mitigated_by', 'affects'],
      dtype=object)

### Build relation2id

In [21]:
relation2id = {}

for idx, rel in enumerate(
    sorted(
        edges["relation"].unique()
    )
):

    relation2id[rel] = idx

### Check Relation Mapping

In [22]:
relation2id

{'affects': 0,
 'has_weakness': 1,
 'mapped_to': 2,
 'maps_to': 3,
 'mitigated_by': 4}

-----------

In [23]:
edges["relation"].value_counts()

relation
maps_to         5845
affects         4772
has_weakness    2296
mitigated_by    1448
mapped_to       1212
Name: count, dtype: int64

In [24]:
cwe_capec.head()

,source,relation,target
0,CWE-276,mapped_to,CAPEC-1
1,CWE-285,mapped_to,CAPEC-1
2,CWE-434,mapped_to,CAPEC-1
3,CWE-693,mapped_to,CAPEC-1
4,CWE-732,mapped_to,CAPEC-1


In [25]:
capec_attack.head()

,source,relation,target
0,CAPEC-10,maps_to,T1574.007
1,CAPEC-10,maps_to,T1480.001
2,CAPEC-101,maps_to,T1055.011
3,CAPEC-101,maps_to,T1583.007
4,CAPEC-101,maps_to,T1583.002


### Save node2id

In [26]:
len(node2id)

7502

In [27]:
with open(
    ml_dir / "node2id.pkl",
    "wb"
) as f:

    pickle.dump(node2id, f)

print("node2id saved")

node2id saved


### Save relation2id

In [28]:
with open(
    ml_dir / "relation2id.pkl",
    "wb"
) as f:

    pickle.dump(relation2id, f)

print("relation2id saved")

relation2id saved


### Verify Files Exist

In [29]:
import os

print(
    os.listdir(ml_dir)
)

['node2id.pkl', 'relation2id.pkl']
